# Packages

In [30]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [31]:
!pip install lightning

In [32]:
import os, time, json, random, copy
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

from dataclasses import dataclass, asdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import (
    Dataset,
    DataLoader,
    random_split,
    Subset
)
import torchvision
import torchvision.transforms as T
from torchvision.datasets import CIFAR10

import lightning as L
from lightning.pytorch.callbacks import (
    ModelCheckpoint,
    LearningRateMonitor,
    EarlyStopping,
    ModelSummary
)
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.tuner import Tuner
from torchmetrics.classification import MulticlassAccuracy

# Data

In [33]:
@dataclass
class CFG:
  # data
  data_root: str = "/content/drive/MyDrive"
  val_frac: float = 0.1           # validation ratio from train dataset
  img_size: int = 224

  # training
  batch_size: int = 64
  num_workers: int = 2            # Colab
  epochs: int = 10
  lr: float = 0.01
  momentum: float = 0.9
  weight_decay: float = 5e-4

  # utils
  seed: int = 42
  amp: bool = True                # mixed precision
  device: str = "cuda" if torch.cuda.is_available() else "cpu"

In [34]:
cfg = CFG()
device = torch.device(cfg.device)
print(json.dumps(asdict(cfg), indent = 2))

{
  "data_root": "/content/drive/MyDrive",
  "val_frac": 0.1,
  "img_size": 224,
  "batch_size": 64,
  "num_workers": 2,
  "epochs": 10,
  "lr": 0.01,
  "momentum": 0.9,
  "weight_decay": 0.0005,
  "seed": 42,
  "amp": true,
  "device": "cuda"
}


In [35]:
L.seed_everything(cfg.seed)
g = torch.Generator().manual_seed(cfg.seed)

INFO: Seed set to 42
INFO:lightning.fabric.utilities.seed:Seed set to 42


In [36]:
train_full = CIFAR10(cfg.data_root, train = True, download = True, transform = None)
test_set = CIFAR10(cfg.data_root, train = False, download = True, transform = None)
print(train_full.data.shape, train_full.data.dtype)

CLASSES = train_full.classes
print(CLASSES)

100%|██████████| 170M/170M [05:30<00:00, 517kB/s]


(50000, 32, 32, 3) uint8
['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [37]:
print(len(train_full))
print(train_full[0])
img, label = train_full[0]
print(type(img), img.size, img.mode, label) # 5만장, 10 클래스, 32 x 32 x 3(RGB) PIL 이미지

50000
(<PIL.Image.Image image mode=RGB size=32x32 at 0x7C7D00A611D0>, 6)
<class 'PIL.Image.Image'> (32, 32) RGB 6


In [38]:
arr = np.array(img)
print(arr.shape)

(32, 32, 3)


In [39]:
n_val = int(cfg.val_frac * len(train_full))
n_train = len(train_full) - n_val
train, val = random_split(train_full, [n_train, n_val], generator = g) #?

print(f"train size: {len(train)}")
print(f"val size: {len(val)}")
print(f"test size: {len(test_set)}")

train size: 45000
val size: 5000
test size: 10000


In [40]:
def build_transform(mean, std):
  train_tf = T.Compose(
      [
          T.RandomCrop(32, padding = 4, padding_mode = "reflect"), # 40 x 40에서 32 x 32로 잘라냄, padding_mode = "constnt": 0, "reflect": "안쪽 픽셀을 거울처럼 반사", "edge": "테두리 픽셀을 복제", ...
          T.RandomHorizontalFlip(p = 0.5), # 좌우 반전, 반전 확률 50%, 1.0 -> 항상 반전 + class를 확인했을 때 verticalflip은 사용하지 않기로 결정(ex. 새, 고양이, 강아지 등 이슈)
          T.ToTensor(), # PIL 데이터를 PyTorch 텐서로 변환 + 스케일링, 스케일링 되어 있으면 T.PILToTensor()
          T.Normalize(mean, std), # 데이터셋 전체를 모았을 때 mean: 0, std: 1
      ]
  )
  eval_tf = T.Compose(
      [
          T.ToTensor(),
          T.Normalize(mean, std),
      ]
  )
  return train_tf, eval_tf

MEAN, STD = [0.4914, 0.4822, 0.4465], [0.2470, 0.2435, 0.2616]
train_tf, eval_tf = build_transform(MEAN, STD)

In [41]:
class TransformDataset(Dataset):
  def __init__(self, base, transform):
    self.base = base
    self.transform = transform

  def __len__(self):
    return len(self.base)

  def __getitem__(self, idx):
    img, label = self.base[idx]
    img = self.transform(img)
    return img, label

In [42]:
ds_train = TransformDataset(train, train_tf)
ds_val = TransformDataset(val, eval_tf)
ds_test = TransformDataset(test_set, eval_tf)

print(len(ds_train), len(ds_val), len(ds_test))

45000 5000 10000


In [43]:
class CIFAR10DataModule(L.LightningDataModule):
  def __init__(
      self,
      ds_train, ds_val, ds_test,
      batch_size: int = 128,
      num_workers: int = 4
  ):
    super().__init__()
    self.ds_train, self.ds_val, self.ds_test = ds_train, ds_val, ds_test
    self.batch_size = batch_size
    self.num_workers = num_workers

  def train_dataloader(self):
    return DataLoader(
        self.ds_train,
        batch_size = self.batch_size,
        num_workers = self.num_workers,
        shuffle = True
    )

  def val_dataloader(self):
    return DataLoader(
        self.ds_val,
        batch_size = self.batch_size,
        num_workers = self.num_workers,
        shuffle = False
    )

  def test_dataloader(self):
    return DataLoader(
        self.ds_test,
        batch_size = self.batch_size,
        num_workers = self.num_workers,
        shuffle = False
    )

In [44]:
dm = CIFAR10DataModule(
    ds_train, ds_val, ds_test,
    batch_size = cfg.batch_size,
    num_workers = cfg.num_workers
)

# Models(32x32)

## AlexNet

In [45]:
class AlexNet(nn.Module):
  def __init__(self, num_classes = 10, dropout = 0.5):
    super().__init__()
    self.features = nn.Sequential(
        nn.Conv2d(3, 96, 3, padding = 1),
        nn.ReLU(inplace = True), # inplace = True: x.clamp_(min = 0) -> x를 직접 수정. 새 메모리 없음, False: y = torch.clamp(x, min = 0) -> 새 메모리 할당 => ResNet에서
        nn.MaxPool2d(2),
        nn.Conv2d(96, 256, 3, padding = 1),
        nn.ReLU(inplace = True),
        nn.MaxPool2d(2),
        nn.Conv2d(256, 384, 3, padding = 1),
        nn.ReLU(inplace = True),
        nn.Conv2d(384, 384, 3, padding = 1),
        nn.ReLU(inplace = True),
        nn.Conv2d(384, 256, 3, padding = 1),
        nn.ReLU(inplace = True),
        nn.MaxPool2d(2)
    )
    self.classifier = nn.Sequential(
        nn.Dropout(p = dropout),
        nn.Linear(256 * 4 * 4, 4096),
        nn.ReLU(inplace = True),
        nn.Dropout(p = dropout),
        nn.Linear(4096, 4096),
        nn.ReLU(inplace = True),
        nn.Linear(4096, num_classes)
    )
  def forward(self, x):
    x = self.features(x)
    x = torch.flatten(x, 1)
    x = self.classifier(x)
    return x

## VGGNet

In [46]:
class VGG16(nn.Module):
  def __init__(self, num_classes = 10):
    super().__init__()
    self.features = nn.Sequential(
        # block1: 32 -> 16
        nn.Conv2d(3, 64, 3, padding = 1),
        nn.BatchNorm2d(64),
        nn.ReLU(inplace = True),
        nn.Conv2d(64, 64, 3, padding = 1),
        nn.BatchNorm2d(64),
        nn.ReLU(inplace = True),
        nn.MaxPool2d(2, 2),

        # block2: 16 -> 8
        nn.Conv2d(64, 128, 3, padding = 1),
        nn.BatchNorm2d(128),
        nn.ReLU(inplace = True),
        nn.Conv2d(128, 128, 3, padding = 1),
        nn.BatchNorm2d(128),
        nn.ReLU(inplace = True),
        nn.MaxPool2d(2, 2),

        # block3: 8 -> 4
        nn.Conv2d(128, 256, 3, padding = 1),
        nn.BatchNorm2d(256),
        nn.ReLU(inplace = True),
        nn.Conv2d(256, 256, 3, padding = 1),
        nn.BatchNorm2d(256),
        nn.ReLU(inplace = True),
        nn.Conv2d(256, 256, 3, padding = 1),
        nn.BatchNorm2d(256),
        nn.ReLU(inplace = True),
        nn.MaxPool2d(2, 2),

        # block4: 4 -> 2
        nn.Conv2d(256, 512, 3, padding = 1),
        nn.BatchNorm2d(512),
        nn.ReLU(inplace = True),
        nn.Conv2d(512, 512, 3, padding = 1),
        nn.BatchNorm2d(512),
        nn.ReLU(inplace = True),
        nn.Conv2d(512, 512, 3, padding = 1),
        nn.BatchNorm2d(512),
        nn.ReLU(inplace = True),
        nn.MaxPool2d(2, 2),

        # block5: 2 -> 2
        nn.Conv2d(512, 512, 3, padding = 1),
        nn.BatchNorm2d(512),
        nn.ReLU(inplace = True),
        nn.Conv2d(512, 512, 3, padding = 1),
        nn.BatchNorm2d(512),
        nn.ReLU(inplace = True),
        nn.Conv2d(512, 512, 3, padding = 1),
        nn.BatchNorm2d(512),
        nn.ReLU(inplace = True),
    )
    self.classifier = nn.Sequential(
        nn.Linear(512 * 2 * 2, 4096),
        nn.ReLU(inplace = True),
        nn.Dropout(),
        nn.Linear(4096, 4096),
        nn.ReLU(inplace = True),
        nn.Dropout(),
        nn.Linear(4096, num_classes)
    )

  def forward(self, x):
    x = self.features(x)
    x = torch.flatten(x, 1)
    x = self.classifier(x)
    return x

## ResNet

In [47]:
class ResBlock(nn.Module):
  def __init__(self, in_ch, out_ch, stride = 1):
    super().__init__()
    self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride, padding = 1, bias = False)
    self.bn1 = nn.BatchNorm2d(out_ch)
    self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, padding = 1, bias = False)
    self.bn2 = nn.BatchNorm2d(out_ch)

    if stride != 1 or in_ch != out_ch:
      self.shortcut = nn.Sequential(
          nn.Conv2d(in_ch, out_ch, 1, stride, bias = False),
          nn.BatchNorm2d(out_ch)
      )
    else:
      self.shortcut = nn.Identity()

  def forward(self, x):
    identity = self.shortcut(x)

    out = self.conv1(x)
    out = self.bn1(out)
    out = F.relu(out)

    out = self.conv2(out)
    out = self.bn2(out)

    out = out + identity
    out = F.relu(out)
    return out


In [48]:
class ResNet(nn.Module):
  def __init__(self, num_classes = 10, block = ResBlock):
    super().__init__()

    self.conv1 = nn.Conv2d(3, 16, 3, stride = 1, padding = 1, bias = False)
    self.bn1 = nn.BatchNorm2d(16)

    self.block1_1 = block(16, 16, 1)
    self.block1_2 = block(16, 16, 1)
    self.block1_3 = block(16, 16, 1)

    self.block2_1 = block(16, 32, 2)
    self.block2_2 = block(32, 32, 1)
    self.block2_3 = block(32, 32, 1)

    self.block3_1 = block(32, 64, 2)
    self.block3_2 = block(64, 64, 1)
    self.block3_3 = block(64, 64, 1)

    self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
    self.fc = nn.Linear(64, num_classes)

  def forward(self, x):
    x = F.relu(self.bn1(self.conv1(x)))

    x = self.block1_1(x)
    x = self.block1_2(x)
    x = self.block1_3(x)

    x = self.block2_1(x)
    x = self.block2_2(x)
    x = self.block2_3(x)

    x = self.block3_1(x)
    x = self.block3_2(x)
    x = self.block3_3(x)

    x = self.avgpool(x)
    x = torch.flatten(x, 1)
    x = self.fc(x)
    return x

# Training

In [49]:
class LitClassifier(L.LightningModule):
  def __init__(
      self,
      model,
      lr: float = 0.1,
      momentum: float = 0.9,
      weight_decay: float = 1e-4,
      max_epochs: int = 30
  ):
    super().__init__()
    self.model = model
    self.lr = lr
    self.momentum = momentum
    self.weight_decay = weight_decay
    self.max_epochs = max_epochs
    self.criterion = nn.CrossEntropyLoss()

  def forward(self, x):
    return self.model(x)

  def _shared_step(self, batch, stage):
    x, y = batch
    logits = self(x)
    loss = self.criterion(logits, y)
    acc = (logits.argmax(dim = -1) == y).float().mean()
    self.log(f"{stage}_loss", loss, on_step = False, on_epoch = True, prog_bar = True, logger = True)
    self.log(f"{stage}_acc", acc, on_step = False, on_epoch = True, prog_bar = True, logger = True)
    return loss

  def training_step(self, batch, batch_idx):
    return self._shared_step(batch, "train")

  def validation_step(self, batch, batch_idx):
    return self._shared_step(batch, "val")

  def test_step(self, batch, batch_idx):
    return self._shared_step(batch, "test")

  def configure_optimizers(self):
    optimizer = torch.optim.SGD(
        self.parameters(),
        lr = self.lr,
        momentum = self.momentum,
        weight_decay = self.weight_decay
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max = self.max_epochs
    )
    return {"optimizer": optimizer, "lr_scheduler": scheduler}

# Evaluation

In [50]:
configs = [
    (AlexNet, "AlexNet",  0.01),   # BN 없음 → lr 낮게
    (VGG16,   "VGG16",    0.05),
    (ResNet,  "ResNet20", 0.1),    # BN 있음 → lr 크게
]

RESULTS = []

for build, name, lr in configs:
  L.seed_everything(cfg.seed, workers = True)

  model = build()
  lit = LitClassifier(
      model = model,
      lr = lr,
      max_epochs = cfg.epochs
  )

  ckpt = ModelCheckpoint(monitor = "val_acc", mode = "max", save_top_k = 1)

  trainer = L.Trainer(
      max_epochs = cfg.epochs,
      accelerator = "auto",
      devices = 1,
      precision = "16-mixed" if cfg.amp else "32-true",
      logger = CSVLogger("logs", name = name),
      callbacks = [
          ckpt,
          LearningRateMonitor(logging_interval = "epoch"),
      ]
  )
  trainer.fit(lit, datamodule = dm)
  test = trainer.test(lit, datamodule = dm, ckpt_path = "best", verbose = False)[0]

  RESULTS.append({
      "model": name,
      "params_M": round(sum(p.numel() for p in model.parameters()) / 1e6, 3),
      "best_val_acc": round(ckpt.best_model_score.item() * 100, 2),
      "test_acc": round(test["test_acc"] * 100, 2),
  })


print(pd.DataFrame(RESULTS).set_index("model"))

INFO: Seed set to 42
INFO:lightning.fabric.utilities.seed:Seed set to 42
INFO: Using 16bit Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: LOCAL_RANK: 0 - C

┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model     │ AlexNet          │ 36.9 M │ train │     0 │
│ 1 │ criterion │ CrossEntropyLoss │      0 │ train │     0 │
└───┴───────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 36.9 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 36.9 M                                                                                               
Total estimated model params size (MB): 147.701                                                                    
Modules in train mode: 24                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

...

Output()

INFO: `Trainer.fit` stopped: `max_epochs=10` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=10` reached.


INFO: Restoring states from the checkpoint path at logs/ResNet20/version_0/checkpoints/epoch=9-step=7040.ckpt
INFO:lightning.pytorch.utilities.rank_zero:Restoring states from the checkpoint path at logs/ResNet20/version_0/checkpoints/epoch=9-step=7040.ckpt
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: Loaded model weights from the checkpoint at logs/ResNet20/version_0/checkpoints/epoch=9-step=7040.ckpt
INFO:lightning.pytorch.utilities.rank_zero:Loaded model weights from the checkpoint at logs/ResNet20/version_0/checkpoints/epoch=9-step=7040.ckpt


Output()

          params_M  best_val_acc  test_acc
model                                     
AlexNet     36.925         81.08     80.68
VGG16       39.938         83.44     83.44
ResNet20     0.272         85.62     84.96
